# IEDB B-cell dataset preparation

In Kaggle, import this notebook, enable **Internet** in Notebook settings, and select **Run All**. A CPU session is sufficient. The notebook downloads the IEDB export and builds CD-HIT 4.8.1 locally without administrator privileges.

The pipeline uses the original label definitions, 80% sequence identity threshold, and seed of 42. Validation and test each contain 5,000 negatives and 5,000 stratified positives by default. All generated files are under `/kaggle/working/iedb_bcell/data`.

Sources: [IEDB database export](https://www.iedb.org/database_export_v3.php), [CD-HIT 4.8.1 release](https://github.com/weizhongli/cdhit/releases/tag/V4.8.1), [CD-HIT build instructions](https://github.com/weizhongli/cdhit/blob/master/README).

In [ ]:
import os
import shutil
import subprocess
import tarfile
from pathlib import Path, PurePosixPath
from zipfile import ZipFile, is_zipfile

import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

WORK_DIR = Path("/kaggle/working/iedb_bcell")
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = WORK_DIR / "downloads"
BUILD_DIR = WORK_DIR / "tools"
for directory in (DATA_DIR, CACHE_DIR, BUILD_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IEDB_URL = (
    "https://www.iedb.org/downloader.php?"
    "file_name=doc/bcell_full_v3_single_file.zip"
)
CDHIT_URL = (
    "https://github.com/weizhongli/cdhit/releases/download/"
    "V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz"
)
# Optional path to an archived copy mounted under /kaggle/input.
LOCAL_IEDB_ZIP = None
HOLDOUT_SIZE_PER_CLASS = 5_000
SEED = 42
CPU_THREADS = min(4, os.cpu_count() or 1)

## Download and extract the IEDB CSV

In [ ]:
def download(url, destination, is_valid):
    if destination.is_file() and is_valid(destination):
        return destination

    partial = destination.with_suffix(destination.suffix + ".part")
    try:
        subprocess.run(
            ["curl", "--fail", "--location", "--show-error", "--retry", "3",
             "--connect-timeout", "30", "--output", str(partial), url],
            check=True,
        )
        if not is_valid(partial):
            raise ValueError(f"Expected an archive, but received a different response from {url}")
        partial.replace(destination)
    finally:
        partial.unlink(missing_ok=True)
    return destination


archive = (
    Path(LOCAL_IEDB_ZIP) if LOCAL_IEDB_ZIP
    else download(IEDB_URL, CACHE_DIR / "bcell_full_v3_single_file.zip", is_zipfile)
)
csv_path = DATA_DIR / "bcell_full_v3.csv"
partial_csv = csv_path.with_suffix(".csv.part")
try:
    with ZipFile(archive) as zipped:
        matches = [
            member for member in zipped.infolist()
            if not member.is_dir() and PurePosixPath(member.filename).name == csv_path.name
        ]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one {csv_path.name} in {archive}")
        with zipped.open(matches[0]) as source, partial_csv.open("wb") as target:
            shutil.copyfileobj(source, target)
    partial_csv.replace(csv_path)
finally:
    partial_csv.unlink(missing_ok=True)
print(f"Using {csv_path} ({csv_path.stat().st_size:,} bytes)")

## Install CD-HIT 4.8.1

Build the upstream release with OpenMP. The `zlib=no` build option is sufficient because the clustering inputs are uncompressed FASTA files.

In [ ]:
cdhit_dir = BUILD_DIR / "cd-hit-v4.8.1-2019-0228"
binaries = {name: cdhit_dir / name for name in ("cd-hit", "cd-hit-2d")}

def expected_version(binary):
    if not binary.is_file() or not os.access(binary, os.X_OK):
        return False
    result = subprocess.run([str(binary), "-h"], capture_output=True, text=True)
    help_text = result.stdout + result.stderr
    return "4.8.1" in help_text and "Usage:" in help_text


if not all(expected_version(binary) for binary in binaries.values()):
    if not all(shutil.which(tool) for tool in ("make", "g++")):
        raise RuntimeError("The CD-HIT source build requires make and g++ in the runtime.")
    source_archive = download(CDHIT_URL, CACHE_DIR / f"{cdhit_dir.name}.tar.gz", tarfile.is_tarfile)
    subprocess.run(["tar", "-xzf", str(source_archive), "-C", str(BUILD_DIR)], check=True)
    subprocess.run(
        ["make", "-j", str(CPU_THREADS), "zlib=no", *binaries],
        cwd=cdhit_dir,
        check=True,
    )

for name, binary in binaries.items():
    if not expected_version(binary):
        raise RuntimeError(f"Could not verify {name} version 4.8.1 at {binary}")
    print(f"{name} 4.8.1: {binary}")

## Curate peptide records and assign labels

The input export has two header rows. The workflow groups assay results by IEDB epitope ID and applies the original uppercase-sequence and 1–25 residue filters. Label 0 requires at least one negative result and zero positive results. Among records with positive results, labels 1, 2, and 3 indicate IgA, IgE, and IgM, respectively, when exactly one of those three isotypes has a positive result. Label 4 covers the remaining positives.

In [ ]:
import csv


def load_iedb_assays(path):
    """Read the six required fields using both rows of the IEDB header."""
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        groups = next(reader, [])
        fields = next(reader, [])

    if not groups or len(groups) != len(fields):
        raise ValueError("Expected two equally wide header rows in the IEDB CSV.")

    # Fill forward group names in exports that omit repeated group names.
    header = []
    current_group = ""
    for group, field in zip(groups, fields):
        current_group = group.strip() or current_group
        header.append((current_group, field.strip()))

    def normalize(value):
        return " ".join(value.lower().replace("_", " ").split())

    normalized = [(normalize(group), normalize(field)) for group, field in header]
    # Use group names to distinguish epitope fields from similarly named assay fields.
    rules = {
        "Epitope IRI": (("Epitope",), ("IEDB IRI", "Epitope IRI")),
        "Object Type": (("Epitope",), ("Object Type",)),
        "Description": (("Epitope",), ("Name", "Description")),
        "Organism Name": (
            ("Epitope", "Epitope Source Organism", "Epitope Source"),
            ("Source Organism", "Organism Name"),
        ),
        "Qualitative Measure": (("Assay",), ("Qualitative Measure",)),
        "Heavy Chain Type": (
            ("Assay Antibody", "Assay - Antibody", "Antibody", "Assay"),
            ("Heavy Chain Isotype", "Heavy Chain Type", "Antibody Isotype"),
        ),
    }
    selected = {}
    unresolved = {}
    for target, (allowed_groups, aliases) in rules.items():
        allowed_groups = {normalize(group) for group in allowed_groups}
        aliases = {normalize(alias) for alias in aliases}
        matches = [
            index for index, (group, field) in enumerate(normalized)
            if group in allowed_groups and field in aliases
        ]
        if not matches:
            # The original notebook selected these exact legacy labels by name.
            matches = [
                index for index, (_, field) in enumerate(normalized)
                if field == normalize(target)
            ]
        if len(matches) == 1:
            selected[matches[0]] = target
        else:
            unresolved[target] = [header[index] for index in matches]

    if unresolved or len(selected) != len(rules):
        formatted_header = "\n".join(f"  {group} / {field}" for group, field in header)
        raise ValueError(
            f"Could not uniquely resolve required IEDB columns: {unresolved}\n"
            f"Detected header pairs:\n{formatted_header}"
        )

    positions = sorted(selected)
    print("Resolved IEDB columns:")
    for index in positions:
        group, field = header[index]
        print(f"  {group} / {field} -> {selected[index]}")

    frame = pd.read_csv(
        path, header=None, skiprows=2, usecols=positions,
        dtype="string", encoding="utf-8-sig",
    )
    return frame.rename(columns=selected)


dfa = load_iedb_assays(csv_path)
print(f"Assay rows: {len(dfa):,}")


dfa = dfa.loc[
    dfa["Object Type"].eq("Linear peptide")
    & dfa["Description"].str.fullmatch(r"[A-Z]+", na=False)
].copy()
print(f"Linear peptide assay rows with uppercase sequences: {len(dfa):,}")


dfa["Epitope ID"] = dfa["Epitope IRI"].str.extract(r"epitope/(\d+)").astype("int")
dfa["Assay Result"] = dfa["Qualitative Measure"].str[:8]
dfa["Isotype"] = dfa["Heavy Chain Type"].str.extract(r"(Ig\w)")


dfe = (
    dfa[["Epitope ID", "Description"]]
    .drop_duplicates()
    .set_index("Epitope ID")
    .sort_index()
)
dfe["Length"] = dfe["Description"].str.len()


dfr = dfa.groupby(["Epitope ID", "Assay Result"]).size().unstack(fill_value=0)


dfg = (
    dfa[dfa["Assay Result"] == "Positive"]
    .groupby(["Epitope ID", "Isotype"])
    .size()
    .unstack(fill_value=0)
)


if not dfe.index.is_unique:
    raise ValueError("An IEDB epitope ID maps to more than one peptide description.")

dfb = dfe.join(dfr).join(dfg)
count_columns = ["Negative", "Positive", "IgA", "IgE", "IgM"]
dfb = dfb.reindex(columns=dfb.columns.union(count_columns, sort=False))
# Missing assay/isotype counts represent zero matching rows.
dfb[count_columns] = dfb[count_columns].fillna(0).astype(int)


ids_sars2 = set(
    dfa.loc[
        dfa["Organism Name"] == "Severe acute respiratory syndrome coronavirus 2",
        "Epitope ID",
    ]
)
dfb["SARS_CoV2"] = 0
dfb.loc[dfb.index.isin(ids_sars2), "SARS_CoV2"] = 1
print(dfb["SARS_CoV2"].value_counts())


df0 = dfb[(dfb["Negative"] >= 1) & (dfb["Positive"] == 0)]
df0 = df0[(df0["Length"] >= 1) & (df0["Length"] <= 25)].copy()
print(f"Negative peptides: {len(df0):,}")


df1 = dfb[dfb["Positive"] >= 1]
df1 = df1[(df1["Length"] >= 1) & (df1["Length"] <= 25)].copy()
print(f"Positive peptides: {len(df1):,}")


# 0 for negative
df0["Label"] = 0
# 1, 2, 3 for Ig
ig_lst = ["IgA", "IgE", "IgM"]
# 4 for other
df1["Label"] = len(ig_lst) + 1
for i, ig in enumerate(ig_lst):
    df1.loc[
        (df1[ig] > 0) & (df1[[ig_ for ig_ in ig_lst if ig_ != ig]] == 0).all(axis=1),
        "Label",
    ] = (
        i + 1
    )

print(df1["Label"].value_counts())


df0[["Description", "Label", "SARS_CoV2"]].to_csv(DATA_DIR / "0.csv")
df1[["Description", "Label", "SARS_CoV2"]].to_csv(DATA_DIR / "123.csv")

## Cluster peptides at 80% identity

First, cluster the negative and positive inputs separately. Then use the positive representatives as database 1 and screen the negative representatives as database 2. The notebook preserves the executable defaults, including CD-HIT's short-sequence cutoff (default `-l 10`). The 1–25 residue filter therefore defines the initial candidates. Use the resulting FASTA files to determine which records survived clustering.

In [ ]:
def write_fasta(frame, filename):
    with (DATA_DIR / filename).open("w") as handle:
        for row in frame.itertuples():
            handle.write(f">{row.Index}_{row.Label}\n{row.Description}\n")


output_columns = ["Description", "Label", "SARS_CoV2"]
df0 = df0[output_columns]
df1 = df1[output_columns]
write_fasta(df0, "0.fasta")
write_fasta(df1, "123.fasta")


def run_cdhit(program, *arguments):
    subprocess.run(
        [str(binaries[program]), *arguments, "-c", "0.8"],
        cwd=DATA_DIR,
        check=True,
    )


run_cdhit("cd-hit", "-i", "0.fasta", "-o", "0r.fasta")
run_cdhit("cd-hit", "-i", "123.fasta", "-o", "123r.fasta")
run_cdhit("cd-hit-2d", "-i", "123r.fasta", "-i2", "0r.fasta", "-o", "0rr.fasta")

## Export representatives and create train, validation, and test splits

In [ ]:
def fasta2id(name):
    ids = set()
    with (DATA_DIR / f"{name}.fasta").open() as f:
        for line in f:
            if line.startswith(">"):
                ids.add(int(line[1:].split("_")[0]))
    return ids

ids_0r = fasta2id("0rr")
ids_123r = fasta2id("123r")
print(len(ids_0r.union(ids_123r)))

df = pd.concat([df0, df1], sort=True)
dfc = df.loc[df.index.isin(ids_0r | ids_123r)]
print(dfc["Label"].value_counts())

dfc[dfc["Label"] == 0].to_csv(DATA_DIR / "0r.csv")
dfc[dfc["Label"] > 0].to_csv(DATA_DIR / "123r.csv")

write_fasta(dfc.loc[dfc["SARS_CoV2"].eq(1)], "sars.fasta")

size = HOLDOUT_SIZE_PER_CLASS
splits = ("train", "valid", "test")

df_neg = pd.read_csv(DATA_DIR / "0r.csv", index_col=0)
df_neg = df_neg[df_neg["SARS_CoV2"] == 0]
df_neg = df_neg.sample(frac=1, random_state=SEED)
if size < 1 or len(df_neg) <= 2 * size:
    raise ValueError(
        f"Need more than {2 * size:,} non-SARS-CoV-2 negatives; found {len(df_neg):,}."
    )
d_neg = dict.fromkeys(splits)
d_neg["train"] = df_neg.iloc[: -2 * size]
d_neg["valid"] = df_neg.iloc[-2 * size : -size]
d_neg["test"] = df_neg.iloc[-size:]

df_pos = pd.read_csv(DATA_DIR / "123r.csv", index_col=0)
df_pos = df_pos[df_pos["SARS_CoV2"] == 0]
if len(df_pos) <= 2 * size:
    raise ValueError(
        f"Need more than {2 * size:,} non-SARS-CoV-2 positives; found {len(df_pos):,}."
    )
y = df_pos["Label"]
sss = StratifiedShuffleSplit(n_splits=1, test_size=size, random_state=SEED)
train_index, test_index = next(sss.split(y, y))
d_pos = dict.fromkeys(splits)
d_pos["test"] = df_pos.iloc[test_index]
df_pos = df_pos.iloc[train_index]
y = df_pos["Label"]
sss = StratifiedShuffleSplit(n_splits=1, test_size=size, random_state=SEED)
train_index, valid_index = next(sss.split(y, y))
d_pos["valid"] = df_pos.iloc[valid_index]
d_pos["train"] = df_pos.iloc[train_index]

for split in splits:
    partition = pd.concat([d_neg[split], d_pos[split]])
    path = DATA_DIR / f"{split}.csv"
    partition.to_csv(path)
    print(f"{path.name}: {len(partition):,} rows | labels: "
          f"{partition['Label'].value_counts().sort_index().to_dict()}")
print(f"Outputs: {DATA_DIR}")